# Pipeline essencial — Predição de ganho EDFA

Este notebook contém **somente** a sequência utilizada para reproduzir a **MLP residual 204→256→256→128→95**. Não inclui FFT, GPR, CatBoost, Ridge, CNN ou ensembles. Construído a partir das células originais do notebook 12, com remoção das dependências de modelos descartados. Requer dados oficiais no Volume indicado e acesso de rede ao repositório público SJTU. **Não foi executado em um ambiente Databricks limpo**, portanto o treinamento ainda precisa ser validado. A comparação com o modelo público SJTU está fora do pipeline essencial.

Etapas: configuração → split COSMOS → 8 épocas → 24 épocas → 48 épocas → folds → CV agrupada → ajuste fino (40 épocas) → auditoria.

## 1. Configuração do ambiente

In [ ]:
# 01 - AMBIENTE, DADOS E PREPROCESSADOR SJTU (VERSAO FIXADA)
from pathlib import Path
import subprocess, sys, importlib
import numpy as np, pandas as pd, torch, sklearn
from torch import nn
from torch.utils.data import DataLoader,TensorDataset

ROOT=Path('/Volumes/workspace/default/ofc2026')
DESTINO=ROOT/'resultados/modelagem_espectral_transferencia'
DESTINO.mkdir(parents=True,exist_ok=True)
SEED=42
N=95
REPO=Path('/tmp/ofc_sjtu_comparacao')
URL='https://github.com/kayneZhu/ofc_ml.git'
COMMIT='61c9788c20ad5052cd5bedc089d1784c504871d2'

if not (REPO/'.git').exists():
    subprocess.run(['git','clone',URL,str(REPO)],check=True)
try:
    subprocess.run(['git','-C',str(REPO),'checkout','--detach',COMMIT],check=True)
except subprocess.CalledProcessError:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',COMMIT],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach',COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == COMMIT
SRC=str(REPO/'src')
if SRC not in sys.path: sys.path.insert(0,SRC)
importlib.invalidate_caches()
from ofc_ml.features import get_feature_columns, create_preprocessor

print('Versoes:',sys.version.split()[0],torch.__version__,sklearn.__version__)
print('Preprocessador SJTU:',COMMIT)
print('Pasta de resultados:',DESTINO)


## 2. Divisão COSMOS: índices reproduzíveis

In [ ]:
# 02 - DIVISAO ORIGINAL COSMOS (seed=42, validacao 5%)
# Equivalente a etapa de divisao que era gerada na celula 02 original.
COSMOS=ROOT/'Features/Train/COSMOS_features.csv'
n_cos=len(pd.read_csv(COSMOS,usecols=['target_gain']))
assert n_cos==138624, f'COSMOS inesperado: {n_cos}'
rng=np.random.default_rng(42)
indices=rng.permutation(n_cos)
n_val=int(0.05*len(indices))
idx_val=indices[:n_val]
idx_train=indices[n_val:]
np.save(DESTINO/'indices_cosmos_train.npy',idx_train)
np.save(DESTINO/'indices_cosmos_val.npy',idx_val)
print('Divisao COSMOS:',len(idx_train),'treino e',len(idx_val),'validacao')


## 3. Pré-treinamento: 8 épocas

In [ ]:
# ============================================================
# 10 - MLP COM ENTRADAS COMPLETAS E PRE-TREINAMENTO COSMOS
# ============================================================

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))

ROOT = Path("/Volumes/workspace/default/ofc2026")
REPO = Path("/tmp/ofc_sjtu_comparacao")
DESTINO = ROOT / "resultados/modelagem_espectral_transferencia"
DESTINO.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO / "src"))

from ofc_ml.features import (
    get_feature_columns,
    create_preprocessor
)

N = 95
label_cols = [
    f"calculated_gain_spectra_{i:02d}" for i in range(N)
]

# ------------------------------------------------------------
# 1. DADOS
# ------------------------------------------------------------

X_cos_v2 = pd.read_csv(
    ROOT / "Features/Train/COSMOS_features.csv"
)

Y_cos_v2 = pd.read_csv(
    ROOT / "Features/Train/COSMOS_labels.csv"
)[label_cols].to_numpy(np.float32)

X_ch_v2 = pd.read_csv(
    ROOT / "Features/Train/train_features.csv"
)

idx_train_v2 = np.load(
    DESTINO / "indices_cosmos_train.npy"
)

idx_val_v2 = np.load(
    DESTINO / "indices_cosmos_val.npy"
)

# ------------------------------------------------------------
# 2. PREPROCESSAMENTO
# ------------------------------------------------------------

cat_cols, num_cols, spec_cols, wss_cols = (
    get_feature_columns(X_cos_v2)
)

preproc_v2 = create_preprocessor(
    num_cols,
    spec_cols,
    cat_cols,
    wss_cols,
    "concat"
)

# Ajuste somente no subconjunto de treino COSMOS.
# Nenhuma observacao Challenge de validacao e utilizada.
preproc_v2.fit(X_cos_v2.iloc[idx_train_v2])

Z_cos_v2 = preproc_v2.transform(
    X_cos_v2
).astype(np.float32)

Z_ch_v2 = preproc_v2.transform(
    X_ch_v2
).astype(np.float32)

M_cos_v2 = X_cos_v2[wss_cols].to_numpy(np.float32)

def referencia_v2(df):
    ganho = df["target_gain"].to_numpy(np.float32)
    tilt = df["target_gain_tilt"].to_numpy(np.float32)

    return (
        ganho[:, None]
        + tilt[:, None] * (47 - np.arange(N)) / 94
    ).astype(np.float32)

R_cos_v2 = np.where(
    M_cos_v2 > 0,
    Y_cos_v2 - referencia_v2(X_cos_v2),
    0.0
).astype(np.float32)

assert Z_cos_v2.shape[1] == Z_ch_v2.shape[1]
assert np.isfinite(Z_cos_v2).all()
assert np.isfinite(Z_ch_v2).all()

print("=== NOVOS ATRIBUTOS ===")
print("COSMOS:", Z_cos_v2.shape)
print("Challenge:", Z_ch_v2.shape)
print("Variaveis numericas:", num_cols)
print("Variaveis categoricas:", cat_cols)
print("Modo espectral: concat")

# ------------------------------------------------------------
# 3. ARQUITETURA MLP CONVENCIONAL
# ------------------------------------------------------------

def criar_mlp_v2(input_dim):
    return nn.Sequential(
        nn.Linear(input_dim, 256),
        nn.GELU(),
        nn.Dropout(0.10),

        nn.Linear(256, 256),
        nn.GELU(),
        nn.Dropout(0.10),

        nn.Linear(256, 128),
        nn.GELU(),

        nn.Linear(128, N)
    )

modelo_v2 = criar_mlp_v2(Z_cos_v2.shape[1])

def loss_v2(pred, real, mask):
    erro = nn.functional.smooth_l1_loss(
        pred, real,
        beta=0.25,
        reduction="none"
    )

    return (
        (erro * mask).sum(dim=1)
        / mask.sum(dim=1).clamp_min(1)
    ).mean()

def loader_v2(indices, batch, shuffle):
    return DataLoader(
        TensorDataset(
            torch.from_numpy(Z_cos_v2[indices]),
            torch.from_numpy(R_cos_v2[indices]),
            torch.from_numpy(M_cos_v2[indices])
        ),
        batch_size=batch,
        shuffle=shuffle,
        num_workers=0
    )

train_loader_v2 = loader_v2(
    idx_train_v2, 512, True
)

val_loader_v2 = loader_v2(
    idx_val_v2, 1024, False
)

# ------------------------------------------------------------
# 4. PRE-TREINAMENTO
# ------------------------------------------------------------

otimizador_v2 = torch.optim.AdamW(
    modelo_v2.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

historico_v2 = []
melhor_mae_v2 = float("inf")
melhor_estado_v2 = None

for epoca in range(8):

    modelo_v2.train()
    soma_loss = 0.0
    n_train = 0

    for xb, rb, mb in train_loader_v2:

        otimizador_v2.zero_grad(set_to_none=True)

        pred = modelo_v2(xb)
        loss = loss_v2(pred, rb, mb)

        loss.backward()
        nn.utils.clip_grad_norm_(
            modelo_v2.parameters(), 1.0
        )
        otimizador_v2.step()

        soma_loss += loss.item() * len(xb)
        n_train += len(xb)

    modelo_v2.eval()
    soma_mae = 0.0
    n_val = 0

    with torch.inference_mode():
        for xb, rb, mb in val_loader_v2:

            pred = modelo_v2(xb)

            mae_obs = (
                (torch.abs(pred - rb) * mb).sum(dim=1)
                / mb.sum(dim=1).clamp_min(1)
            )

            soma_mae += mae_obs.sum().item()
            n_val += len(xb)

    mae_val = soma_mae / n_val
    loss_train = soma_loss / n_train

    historico_v2.append({
        "epoca": epoca + 1,
        "loss_train": loss_train,
        "mae_cosmos_val": mae_val
    })

    if mae_val < melhor_mae_v2:
        melhor_mae_v2 = mae_val
        melhor_estado_v2 = {
            k: v.detach().cpu().clone()
            for k, v in modelo_v2.state_dict().items()
        }

    print(
        f"Epoca {epoca+1:02d} | "
        f"Loss: {loss_train:.6f} | "
        f"MAE COSMOS: {mae_val:.6f}"
    )

modelo_v2.load_state_dict(melhor_estado_v2)

# ------------------------------------------------------------
# 5. SALVAR RESULTADOS
# ------------------------------------------------------------

torch.save({
    "model_state_dict": modelo_v2.state_dict(),
    "input_dim": Z_cos_v2.shape[1],
    "output_dim": N,
    "best_mae_cosmos": melhor_mae_v2,
    "feature_mode": "SJTU concat - MLP convencional"
}, DESTINO / "mlp_v2_pretrained.pt")

pd.DataFrame(historico_v2).to_csv(
    DESTINO / "historico_mlp_v2.csv",
    index=False
)

np.save(
    DESTINO / "features_challenge_mlp_v2.npy",
    Z_ch_v2
)

print("\n=== RESULTADO FINAL ===")
print(f"Melhor MAE COSMOS: {melhor_mae_v2:.6f}")
print("Modelo MLP v2 salvo.")
print("Nenhum dado TestGround utilizado.")


## 4. Continuação do pré-treinamento: 24 épocas

In [ ]:
# ============================================================
# 14 - MLP V2: ESTENDER PRE-TREINAMENTO COSMOS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from ofc_ml.features import (
    get_feature_columns,
    create_preprocessor
)

ROOT = Path("/Volumes/workspace/default/ofc2026")
DESTINO = ROOT / "resultados/modelagem_espectral_transferencia"

N = 95
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))

label_cols = [
    f"calculated_gain_spectra_{i:02d}"
    for i in range(N)
]

# ------------------------------------------------------------
# 1. CARREGAR COSMOS
# ------------------------------------------------------------

X = pd.read_csv(
    ROOT / "Features/Train/COSMOS_features.csv"
)

Y = pd.read_csv(
    ROOT / "Features/Train/COSMOS_labels.csv"
)[label_cols].to_numpy(np.float32)

idx_train = np.load(
    DESTINO / "indices_cosmos_train.npy"
)

idx_val = np.load(
    DESTINO / "indices_cosmos_val.npy"
)

cat_cols, num_cols, spec_cols, wss_cols = (
    get_feature_columns(X)
)

preproc = create_preprocessor(
    num_cols,
    spec_cols,
    cat_cols,
    wss_cols,
    "concat"
)

preproc.fit(X.iloc[idx_train])

Z = preproc.transform(X).astype(np.float32)
M = X[wss_cols].to_numpy(np.float32)

referencia = (
    X["target_gain"].to_numpy(np.float32)[:, None]
    + X["target_gain_tilt"].to_numpy(np.float32)[:, None]
    * (47 - np.arange(N)) / 94
).astype(np.float32)

R = np.where(M > 0, Y - referencia, 0).astype(np.float32)

# ------------------------------------------------------------
# 2. RECUPERAR CHECKPOINT DE 8 EPOCAS
# ------------------------------------------------------------

checkpoint = torch.load(
    DESTINO / "mlp_v2_pretrained.pt",
    map_location="cpu",
    weights_only=True
)

def criar_mlp_v2():
    return nn.Sequential(
        nn.Linear(204, 256),
        nn.GELU(),
        nn.Dropout(0.10),

        nn.Linear(256, 256),
        nn.GELU(),
        nn.Dropout(0.10),

        nn.Linear(256, 128),
        nn.GELU(),

        nn.Linear(128, 95)
    )

modelo = criar_mlp_v2()

modelo.load_state_dict(
    checkpoint["model_state_dict"]
)

print("Checkpoint de 8 epocas carregado.")
print(
    "MAE inicial:",
    checkpoint["best_mae_cosmos"]
)

# ------------------------------------------------------------
# 3. DATA LOADERS
# ------------------------------------------------------------

def criar_loader(indices, batch_size, shuffle):
    return DataLoader(
        TensorDataset(
            torch.from_numpy(Z[indices]),
            torch.from_numpy(R[indices]),
            torch.from_numpy(M[indices])
        ),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0
    )

loader_train = criar_loader(idx_train, 512, True)
loader_val = criar_loader(idx_val, 1024, False)

def loss_mascarada(pred, real, mask):
    erro = nn.functional.smooth_l1_loss(
        pred, real,
        beta=0.25,
        reduction="none"
    )

    return (
        (erro * mask).sum(dim=1)
        / mask.sum(dim=1).clamp_min(1)
    ).mean()

# ------------------------------------------------------------
# 4. TREINAMENTO ADICIONAL
# ------------------------------------------------------------

# Taxa menor para refinar o modelo.
# O otimizador e reiniciado, mas os pesos sao preservados.

optimizer = torch.optim.AdamW(
    modelo.parameters(),
    lr=0.0003,
    weight_decay=0.0001
)

melhor_mae = float(
    checkpoint["best_mae_cosmos"]
)

melhor_estado = {
    k: v.detach().cpu().clone()
    for k, v in modelo.state_dict().items()
}

melhor_epoca = 8
sem_melhora = 0
paciencia = 6

historico = []

for epoca in range(9, 25):

    modelo.train()
    soma_loss = 0.0
    qtd_train = 0

    for xb, rb, mb in loader_train:

        optimizer.zero_grad(set_to_none=True)

        pred = modelo(xb)
        loss = loss_mascarada(pred, rb, mb)

        loss.backward()

        nn.utils.clip_grad_norm_(
            modelo.parameters(), 1.0
        )

        optimizer.step()

        soma_loss += loss.item() * len(xb)
        qtd_train += len(xb)

    modelo.eval()
    soma_mae = 0.0
    qtd_val = 0

    with torch.inference_mode():

        for xb, rb, mb in loader_val:

            pred = modelo(xb)

            mae_obs = (
                (torch.abs(pred - rb) * mb).sum(dim=1)
                / mb.sum(dim=1).clamp_min(1)
            )

            soma_mae += mae_obs.sum().item()
            qtd_val += len(xb)

    mae_val = soma_mae / qtd_val
    loss_train = soma_loss / qtd_train

    historico.append({
        "epoca": epoca,
        "loss_train": loss_train,
        "mae_val": mae_val
    })

    if mae_val < melhor_mae - 1e-5:

        melhor_mae = mae_val
        melhor_epoca = epoca
        sem_melhora = 0

        melhor_estado = {
            k: v.detach().cpu().clone()
            for k, v in modelo.state_dict().items()
        }

    else:
        sem_melhora += 1

    print(
        f"Epoca {epoca:02d} | "
        f"Loss: {loss_train:.6f} | "
        f"MAE COSMOS: {mae_val:.6f}"
    )

    if sem_melhora >= paciencia:
        print("Early stopping acionado.")
        break

# ------------------------------------------------------------
# 5. SALVAR MELHOR CHECKPOINT
# ------------------------------------------------------------

modelo.load_state_dict(melhor_estado)

torch.save({
    "model_state_dict": melhor_estado,
    "input_dim": 204,
    "output_dim": 95,
    "best_mae_cosmos": melhor_mae,
    "best_epoch": melhor_epoca
}, DESTINO / "mlp_v2_pretrained_extended.pt")

pd.DataFrame(historico).to_csv(
    DESTINO / "historico_pretrain_extended.csv",
    index=False
)

print("\n=== RESULTADO FINAL ===")
print("MAE anterior: 0.051365")
print(f"Melhor MAE novo: {melhor_mae:.6f}")
print(f"Melhor epoca: {melhor_epoca}")
print("Checkpoint estendido salvo.")


## 5. Continuação do pré-treinamento: até 48 épocas

In [ ]:
# ============================================================
# 17 - MLP V2: CONTINUAR PRE-TREINAMENTO ATE 48 EPOCAS
# ============================================================

from pathlib import Path
import sys
import subprocess
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

ROOT = Path("/Volumes/workspace/default/ofc2026")
DESTINO = ROOT / "resultados/modelagem_espectral_transferencia"
REPO = Path("/tmp/ofc_sjtu_comparacao")
SEED = 42
N = 95

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))

# Recuperar o preprocessador se a sessao foi reiniciada
if not (REPO / "src/ofc_ml/features.py").is_file():
    if REPO.exists():
        import shutil
        shutil.rmtree(REPO)

    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/kayneZhu/ofc_ml.git",
        str(REPO)
    ], check=True)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from ofc_ml.features import (
    get_feature_columns,
    create_preprocessor
)

label_cols = [
    f"calculated_gain_spectra_{i:02d}"
    for i in range(N)
]

# ------------------------------------------------------------
# 1. DADOS COSMOS
# ------------------------------------------------------------

X = pd.read_csv(
    ROOT / "Features/Train/COSMOS_features.csv"
)

Y = pd.read_csv(
    ROOT / "Features/Train/COSMOS_labels.csv"
)[label_cols].to_numpy(np.float32)

idx_train = np.load(
    DESTINO / "indices_cosmos_train.npy"
)

idx_val = np.load(
    DESTINO / "indices_cosmos_val.npy"
)

cat_cols, num_cols, spec_cols, wss_cols = (
    get_feature_columns(X)
)

preproc = create_preprocessor(
    num_cols, spec_cols, cat_cols,
    wss_cols, "concat"
)

preproc.fit(X.iloc[idx_train])

Z = preproc.transform(X).astype(np.float32)
M = X[wss_cols].to_numpy(np.float32)

ref = (
    X["target_gain"].to_numpy(np.float32)[:, None]
    + X["target_gain_tilt"].to_numpy(np.float32)[:, None]
    * (47 - np.arange(N)) / 94
).astype(np.float32)

R = np.where(M > 0, Y - ref, 0.0).astype(np.float32)

assert Z.shape == (138624, 204)
assert np.isfinite(Z).all()

# ------------------------------------------------------------
# 2. CARREGAR CHECKPOINT DE 24 EPOCAS
# ------------------------------------------------------------

ckpt = torch.load(
    DESTINO / "mlp_v2_pretrained_extended.pt",
    map_location="cpu",
    weights_only=True
)

modelo = nn.Sequential(
    nn.Linear(204, 256),
    nn.GELU(),
    nn.Dropout(0.10),

    nn.Linear(256, 256),
    nn.GELU(),
    nn.Dropout(0.10),

    nn.Linear(256, 128),
    nn.GELU(),

    nn.Linear(128, N)
)

modelo.load_state_dict(ckpt["model_state_dict"])

print("Checkpoint inicial carregado.")
print("Epoca:", ckpt["best_epoch"])
print("MAE inicial:", ckpt["best_mae_cosmos"])

# ------------------------------------------------------------
# 3. PREPARAR DATA LOADERS
# ------------------------------------------------------------

def criar_loader(indices, batch_size, shuffle):
    return DataLoader(
        TensorDataset(
            torch.from_numpy(Z[indices]),
            torch.from_numpy(R[indices]),
            torch.from_numpy(M[indices])
        ),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0
    )

train_loader = criar_loader(idx_train, 512, True)
val_loader = criar_loader(idx_val, 1024, False)

def perda_mascarada(pred, real, mask):
    erro = nn.functional.smooth_l1_loss(
        pred, real,
        beta=0.25,
        reduction="none"
    )

    return (
        (erro * mask).sum(dim=1)
        / mask.sum(dim=1).clamp_min(1)
    ).mean()

# ------------------------------------------------------------
# 4. CONTINUAR TREINAMENTO
# ------------------------------------------------------------

# Mesmo LR da etapa de 9 a 24.
# O estado do otimizador nao havia sido salvo;
# aqui reiniciamos AdamW, preservando os pesos.

optimizer = torch.optim.AdamW(
    modelo.parameters(),
    lr=0.0003,
    weight_decay=0.0001
)

melhor_mae = float(ckpt["best_mae_cosmos"])
melhor_epoca = int(ckpt["best_epoch"])

melhor_estado = {
    k: v.detach().cpu().clone()
    for k, v in modelo.state_dict().items()
}

paciencia = 8
sem_melhora = 0
historico = []

for epoca in range(25, 49):

    modelo.train()
    perda_total = 0.0
    qtd_train = 0

    for xb, rb, mb in train_loader:

        optimizer.zero_grad(set_to_none=True)

        pred = modelo(xb)
        loss = perda_mascarada(pred, rb, mb)

        loss.backward()
        nn.utils.clip_grad_norm_(
            modelo.parameters(), 1.0
        )

        optimizer.step()

        perda_total += loss.item() * len(xb)
        qtd_train += len(xb)

    modelo.eval()
    erro_total = 0.0
    qtd_val = 0

    with torch.inference_mode():
        for xb, rb, mb in val_loader:

            pred = modelo(xb)

            erros = (
                (torch.abs(pred - rb) * mb).sum(dim=1)
                / mb.sum(dim=1).clamp_min(1)
            )

            erro_total += erros.sum().item()
            qtd_val += len(xb)

    mae_val = erro_total / qtd_val
    loss_train = perda_total / qtd_train

    historico.append({
        "epoca": epoca,
        "loss_train": loss_train,
        "mae_cosmos_val": mae_val
    })

    if mae_val < melhor_mae - 1e-5:

        melhor_mae = mae_val
        melhor_epoca = epoca
        sem_melhora = 0

        melhor_estado = {
            k: v.detach().cpu().clone()
            for k, v in modelo.state_dict().items()
        }
    else:
        sem_melhora += 1

    print(
        f"Epoca {epoca:02d} | "
        f"Loss: {loss_train:.6f} | "
        f"MAE COSMOS: {mae_val:.6f}"
    )

    if sem_melhora >= paciencia:
        print("Early stopping acionado.")
        break

# ------------------------------------------------------------
# 5. SALVAR SEM SOBRESCREVER MODELOS ANTERIORES
# ------------------------------------------------------------

torch.save({
    "model_state_dict": melhor_estado,
    "input_dim": 204,
    "output_dim": 95,
    "best_epoch": melhor_epoca,
    "best_mae_cosmos": melhor_mae
}, DESTINO / "mlp_v2_pretrained_48ep.pt")

pd.DataFrame(historico).to_csv(
    DESTINO / "historico_pretrain_48ep.csv",
    index=False
)

print("\n=== RESULTADO FINAL ===")
print("MAE 24 epocas: 0.046427")
print(f"Melhor MAE novo: {melhor_mae:.6f}")
print(f"Melhor epoca: {melhor_epoca}")

print("\nCheckpoint salvo:")
print(DESTINO / "mlp_v2_pretrained_48ep.pt")
print("Nenhum rotulo TestGround utilizado.")


## 6. Folds originais do Challenge, sem o modelo FFT

In [ ]:
# 06 - REPRODUZIR A DIVISAO AGRUPADA ORIGINAL (SEM FFT)
# Logica extraida dos grupos da celula 05 do notebook 06 original.
# Apenas os identificadores de fold sao usados: nenhum modelo Fourier/boosting.
X_fold=pd.read_csv(ROOT/'Features/Train/train_features.csv')
GRUPO_COLS=['Category','EDFA_type','edfa_index']
gr_unicos=X_fold[GRUPO_COLS].drop_duplicates()
rng=np.random.default_rng(42)
mapa={}
for categoria,dados in gr_unicos.groupby('Category'):
    chaves=list(dados.itertuples(index=False,name=None))
    rng.shuffle(chaves)
    for j,chave in enumerate(chaves):mapa[chave]=j%4
fold_amostra=np.array([mapa[chave] for chave in X_fold[GRUPO_COLS].itertuples(index=False,name=None)])
assert len(fold_amostra)==490 and set(fold_amostra)==set(range(4))
assert all(len(set(X_fold.loc[fold_amostra==f,GRUPO_COLS].itertuples(index=False,name=None)).intersection(set(X_fold.loc[fold_amostra!=f,GRUPO_COLS].itertuples(index=False,name=None))))==0 for f in range(4))
pd.DataFrame({'indice_original':np.arange(len(fold_amostra)), 'fold':fold_amostra}).to_csv(DESTINO/'folds_challenge.csv',index=False)
print(pd.crosstab(fold_amostra,X_fold['Category'],rownames=['Fold']))


## 7. Validação cruzada agrupada: apenas a MLP final

In [ ]:
# ============================================================
# 18 - MLP V2: VALIDACAO CRUZADA DO CHECKPOINT 48 EPOCAS
# COMPARACAO PAREADA COM O CHECKPOINT DE 24 EPOCAS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path("/Volumes/workspace/default/ofc2026")
DESTINO = ROOT / "resultados/modelagem_espectral_transferencia"

N = 95
SEED = 42

torch.set_num_threads(min(4, torch.get_num_threads()))

label_cols = [
    f"calculated_gain_spectra_{i:02d}"
    for i in range(N)
]

mask_cols = [
    f"DUT_WSS_activated_channel_index_{i:02d}"
    for i in range(N)
]

# ------------------------------------------------------------
# 1. CARREGAR CHALLENGE E ATRIBUTOS
# ------------------------------------------------------------

X = pd.read_csv(
    ROOT / "Features/Train/train_features.csv"
)

Y = pd.read_csv(
    ROOT / "Features/Train/train_labels.csv"
)[label_cols].to_numpy(np.float32)

Z = np.load(
    DESTINO / "features_challenge_mlp_v2.npy"
).astype(np.float32)

M = X[mask_cols].to_numpy(np.float32)
cat = X["Category"].to_numpy()

ref = (
    X["target_gain"].to_numpy(np.float32)[:, None]
    + X["target_gain_tilt"].to_numpy(np.float32)[:, None]
    * (47 - np.arange(N)) / 94
).astype(np.float32)

R = np.where(
    M > 0, Y - ref, 0.0
).astype(np.float32)

especial = (
    X["EDFA_output_power_total"].to_numpy(float)
    == -50
)

# 2. FOLDS AGRUPADOS SALVOS NA ETAPA ANTERIOR
cv=pd.read_csv(DESTINO/'folds_challenge.csv')
folds=cv.sort_values('indice_original')['fold'].to_numpy(int)
assert Z.shape==(490,204) and len(folds)==len(X)
assert (M.sum(axis=1)>0).all()

# ------------------------------------------------------------
# 3. CARREGAR CHECKPOINT DE 48 EPOCAS
# ------------------------------------------------------------

ckpt = torch.load(
    DESTINO / "mlp_v2_pretrained_48ep.pt",
    map_location="cpu",
    weights_only=True
)

print("=== CHECKPOINT COSMOS ===")
print("Melhor epoca:", ckpt["best_epoch"])
print("MAE COSMOS:", ckpt["best_mae_cosmos"])

def criar_mlp():
    return nn.Sequential(
        nn.Linear(204, 256),
        nn.GELU(),
        nn.Dropout(0.10),

        nn.Linear(256, 256),
        nn.GELU(),
        nn.Dropout(0.10),

        nn.Linear(256, 128),
        nn.GELU(),

        nn.Linear(128, N)
    )

def perda_mascarada(pred, real, mask):
    erro = nn.functional.smooth_l1_loss(
        pred,
        real,
        beta=0.25,
        reduction="none"
    )

    return (
        (erro * mask).sum(dim=1)
        / mask.sum(dim=1).clamp_min(1)
    ).mean()

def mae_obs(P):
    return (
        np.where(M > 0, np.abs(Y - P), 0.0).sum(axis=1)
        / M.sum(axis=1)
    )

# ------------------------------------------------------------
# 4. AJUSTE FINO NOS QUATRO FOLDS
# ------------------------------------------------------------

P_48 = ref.copy()

for fold in sorted(np.unique(folds)):

    torch.manual_seed(SEED + int(fold))
    np.random.seed(SEED + int(fold))

    idx_train = np.where(folds != fold)[0]
    idx_val = np.where(folds == fold)[0]

    modelo = criar_mlp()

    modelo.load_state_dict(
        ckpt["model_state_dict"],
        strict=True
    )

    loader = DataLoader(
        TensorDataset(
            torch.from_numpy(Z[idx_train]),
            torch.from_numpy(R[idx_train]),
            torch.from_numpy(M[idx_train])
        ),
        batch_size=64,
        shuffle=True,
        num_workers=0
    )

    optimizer = torch.optim.AdamW(
        modelo.parameters(),
        lr=0.0003,
        weight_decay=0.001
    )

    for epoca in range(40):

        modelo.train()

        for xb, rb, mb in loader:

            optimizer.zero_grad(set_to_none=True)

            pred = modelo(xb)
            loss = perda_mascarada(pred, rb, mb)

            loss.backward()

            nn.utils.clip_grad_norm_(
                modelo.parameters(), 1.0
            )

            optimizer.step()

    modelo.eval()

    with torch.inference_mode():
        residuo = modelo(
            torch.from_numpy(Z[idx_val])
        ).numpy()

    P_48[idx_val] += residuo

    erros_fold = mae_obs(P_48)[idx_val]

    print(
        f"Fold {fold} | "
        f"Treino: {len(idx_train)} | "
        f"Validacao: {len(idx_val)} | "
        f"MAE: {erros_fold.mean():.6f}"
    )


# 5. MAE POR OBSERVACAO E CENARIO; NENHUMA DEPENDENCIA DE MODELO ANTIGO
mae_48=mae_obs(P_48)
print('MAE CV Global:',mae_48.mean())
print(pd.DataFrame({'Category':cat,'MAE':mae_48}).groupby('Category')['MAE'].agg(['count','mean']))
pd.DataFrame({'indice_original':np.arange(len(X)), 'fold':folds, 'MAE':mae_48}).to_csv(DESTINO/'avaliacao_cv_mlp_v2_48ep.csv',index=False)


## 8. Ajuste fino (40 épocas) e avaliação do TestGround

In [ ]:
# ============================================================
# 20 - TESTGROUND: MLP 24 VS 48 VS ENSEMBLE
# ============================================================

from pathlib import Path
import sys
import subprocess
import shutil
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path("/Volumes/workspace/default/ofc2026")
DESTINO = ROOT / "resultados/modelagem_espectral_transferencia"
REPO = Path("/tmp/ofc_sjtu_comparacao")
N = 95

np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(min(4, torch.get_num_threads()))

# ------------------------------------------------------------
# 1. GARANTIR ACESSO AO PREPROCESSADOR SJTU
# ------------------------------------------------------------

if not (REPO / "src/ofc_ml/features.py").is_file():

    if REPO.exists():
        shutil.rmtree(REPO)

    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/kayneZhu/ofc_ml.git",
        str(REPO)
    ], check=True)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from ofc_ml.features import (
    get_feature_columns,
    create_preprocessor
)

label_cols = [
    f"calculated_gain_spectra_{i:02d}"
    for i in range(N)
]

mask_cols = [
    f"DUT_WSS_activated_channel_index_{i:02d}"
    for i in range(N)
]

# ------------------------------------------------------------
# 2. CARREGAR DADOS
# ------------------------------------------------------------

X_cos = pd.read_csv(
    ROOT / "Features/Train/COSMOS_features.csv"
)

X_train = pd.read_csv(
    ROOT / "Features/Train/train_features.csv"
)

X_test = pd.read_csv(
    ROOT / "Features/TestGround/test_features.csv"
)

Y_train = pd.read_csv(
    ROOT / "Features/Train/train_labels.csv"
)[label_cols].to_numpy(np.float32)

Y_test = pd.read_csv(
    ROOT / "Features/TestGround/test_labels.csv"
)[label_cols].to_numpy(float)

idx_cos_train = np.load(
    DESTINO / "indices_cosmos_train.npy"
)

# ------------------------------------------------------------
# 3. RECONSTRUIR PREPROCESSAMENTO
# ------------------------------------------------------------

cat_cols, num_cols, spec_cols, wss_cols = (
    get_feature_columns(X_cos)
)

preproc = create_preprocessor(
    num_cols, spec_cols, cat_cols,
    wss_cols, "concat"
)

preproc.fit(X_cos.iloc[idx_cos_train])

Z_train = preproc.transform(
    X_train
).astype(np.float32)

Z_test = preproc.transform(
    X_test
).astype(np.float32)

Z_anterior = np.load(
    DESTINO / "features_challenge_mlp_v2.npy"
)

assert Z_train.shape == (490, 204)
assert Z_test.shape == (21056, 204)
assert np.allclose(Z_train, Z_anterior, atol=1e-4)

del X_cos

print("Preprocessamento validado.")

# ------------------------------------------------------------
# 4. REFERENCIA FISICA
# ------------------------------------------------------------

def referencia(df):
    return (
        df["target_gain"].to_numpy(np.float32)[:, None]
        + df["target_gain_tilt"].to_numpy(np.float32)[:, None]
        * (47 - np.arange(N)) / 94
    ).astype(np.float32)

ref_train = referencia(X_train)
ref_test = referencia(X_test)

M_train = X_train[mask_cols].to_numpy(np.float32)
M_test = X_test[mask_cols].to_numpy(bool)

R_train = np.where(
    M_train > 0,
    Y_train - ref_train,
    0.0
).astype(np.float32)

# ------------------------------------------------------------
# 5. MLP V2 - CHECKPOINT DE 48 EPOCAS
# ------------------------------------------------------------

modelo = nn.Sequential(
    nn.Linear(204, 256),
    nn.GELU(),
    nn.Dropout(0.10),

    nn.Linear(256, 256),
    nn.GELU(),
    nn.Dropout(0.10),

    nn.Linear(256, 128),
    nn.GELU(),

    nn.Linear(128, N)
)

ckpt = torch.load(
    DESTINO / "mlp_v2_pretrained_48ep.pt",
    map_location="cpu",
    weights_only=True
)

modelo.load_state_dict(ckpt["model_state_dict"])

print(
    "Checkpoint carregado | Epoca:",
    ckpt["best_epoch"]
)

def loss_mascarada(pred, real, mask):
    erro = nn.functional.smooth_l1_loss(
        pred, real, beta=0.25, reduction="none"
    )
    return (
        (erro * mask).sum(dim=1)
        / mask.sum(dim=1).clamp_min(1)
    ).mean()

# ------------------------------------------------------------
# 6. AJUSTE FINO COM TODO O CHALLENGE
# ------------------------------------------------------------

loader = DataLoader(
    TensorDataset(
        torch.from_numpy(Z_train),
        torch.from_numpy(R_train),
        torch.from_numpy(M_train)
    ),
    batch_size=64,
    shuffle=True,
    num_workers=0
)

optimizer = torch.optim.AdamW(
    modelo.parameters(),
    lr=0.0003,
    weight_decay=0.001
)

for epoca in range(40):

    modelo.train()

    for xb, rb, mb in loader:

        optimizer.zero_grad(set_to_none=True)

        pred = modelo(xb)
        loss = loss_mascarada(pred, rb, mb)

        loss.backward()

        nn.utils.clip_grad_norm_(
            modelo.parameters(), 1.0
        )

        optimizer.step()

    if (epoca + 1) % 10 == 0:
        print(f"Ajuste fino: {epoca + 1}/40")

# ------------------------------------------------------------
# 7. INFERENCIA NO TESTGROUND
# ------------------------------------------------------------

modelo.eval()
partes = []

with torch.inference_mode():
    for inicio in range(0, len(Z_test), 512):
        xb = torch.from_numpy(
            Z_test[inicio:inicio + 512]
        )
        partes.append(modelo(xb).numpy())

P48 = ref_test + np.vstack(partes)


# 8. AVALIACAO DA MLP FINAL: media dos MAEs por observacao
assert P48.shape==(21056,95)
assert np.isfinite(P48[M_test]).all()
mae_por_obs=(np.where(M_test,np.abs(Y_test-P48),0.0).sum(axis=1) / M_test.sum(axis=1))
mae_final=float(mae_por_obs.mean())
print('MAE global TestGround:',format(mae_final,'.9f'),'dB')
print('MAE por cenario:')
print(pd.DataFrame({'Category':X_test['Category'],'MAE':mae_por_obs}).groupby('Category')['MAE'].agg(['size','mean']))
print('MAE Public/Private:')
print(pd.DataFrame({'Usage':X_test['Usage'],'MAE':mae_por_obs}).groupby('Usage')['MAE'].agg(['size','mean']))
assert abs(mae_final-0.124232492)<1e-5, 'Resultado divergiu do valor auditado; verificar dados/versoes.'

# Mesmos nomes de artefatos da auditoria original.
pd.DataFrame(P48,columns=label_cols).to_csv(DESTINO/'predicoes_mlp_v2_48ep_testground.csv',index=False)
torch.save({'model_state_dict':modelo.state_dict(),'input_dim':204,'output_dim':95,'pretrain_epochs':48,'finetune_epochs':40},DESTINO/'mlp_v2_48ep_finetuned.pt')
pd.DataFrame({'Category':X_test['Category'],'Usage':X_test['Usage'],'MAE_MLP':mae_por_obs}).to_csv(DESTINO/'avaliacao_mlp_final.csv',index=False)
print('Modelo e previsoes finais salvos em:',DESTINO)


## 9. Auditoria de checkpoint e previsões

In [ ]:
# 09 - AUDITORIA FINAL INDEPENDENTE DO ESTADO DAS CELULAS DE TREINAMENTO
import hashlib,json
from ofc_ml.features import get_feature_columns,create_preprocessor

cos=pd.read_csv(ROOT/'Features/Train/COSMOS_features.csv')
ch=pd.read_csv(ROOT/'Features/Train/train_features.csv')
test=pd.read_csv(ROOT/'Features/TestGround/test_features.csv')
y=pd.read_csv(ROOT/'Features/TestGround/test_labels.csv')[[f'calculated_gain_spectra_{i:02d}' for i in range(95)]].to_numpy(float)
idx=np.load(DESTINO/'indices_cosmos_train.npy')
cat_cols,num_cols,spec_cols,wss_cols=get_feature_columns(cos)
preproc=create_preprocessor(num_cols,spec_cols,cat_cols,wss_cols,'concat')
preproc.fit(cos.iloc[idx])
zch=preproc.transform(ch).astype(np.float32)
ztest=preproc.transform(test).astype(np.float32)
assert zch.shape==(490,204) and ztest.shape==(21056,204)
salvos=np.load(DESTINO/'features_challenge_mlp_v2.npy')
dif_atributos=float(np.abs(zch-salvos).max())
assert dif_atributos<1e-4

mlp=nn.Sequential(nn.Linear(204,256),nn.GELU(),nn.Dropout(.1),nn.Linear(256,256),nn.GELU(),nn.Dropout(.1),nn.Linear(256,128),nn.GELU(),nn.Linear(128,95))
ckpt_path=DESTINO/'mlp_v2_48ep_finetuned.pt'
ckpt=torch.load(ckpt_path,map_location='cpu',weights_only=True)
mlp.load_state_dict(ckpt['model_state_dict'])
mlp.eval()
with torch.inference_mode():
    rp=np.vstack([mlp(torch.from_numpy(ztest[j:j+512])).numpy() for j in range(0,len(ztest),512)])
ref=test['target_gain'].to_numpy(np.float32)[:,None]+test['target_gain_tilt'].to_numpy(np.float32)[:,None]*(47-np.arange(95))/94
p=ref+rp
p_salvas=pd.read_csv(DESTINO/'predicoes_mlp_v2_48ep_testground.csv')[[f'calculated_gain_spectra_{i:02d}' for i in range(95)]].to_numpy(float)
dif_previsoes=float(np.abs(p-p_salvas).max())
assert dif_previsoes<1e-4
mask=test[wss_cols].to_numpy(bool)
mae=float((np.where(mask,np.abs(p-y),0).sum(axis=1)/mask.sum(axis=1)).mean())
assert abs(mae-0.124232492)<1e-5
sha=hashlib.sha256(ckpt_path.read_bytes()).hexdigest()
manifesto={'mae_testground':mae,'diferenca_maxima_features':dif_atributos,'diferenca_maxima_previsoes':dif_previsoes,'checkpoint_sha256':sha,'commit_sjtu':COMMIT,'n_obs':len(test),'n_canais':95}
(DESTINO/'manifesto_reproducao_essencial.json').write_text(json.dumps(manifesto,indent=2))
print('AUDITORIA APROVADA:')
print(json.dumps(manifesto,indent=2))


## 10. Figuras do relatório (1 a 6)

**Execute esta seção depois do treinamento e da auditoria.** As Figuras 1–3 são esquemas conceituais. A Figura 4 utiliza os históricos COSMOS salvos no Volume; as Figuras 5 e 6 utilizam as previsões da MLP final e os rótulos verdadeiros do TestGround online.

**Não é necessário treinar novamente** caso os históricos, as previsões e os datasets já estejam disponíveis no mesmo Volume. As figuras serão exibidas abaixo das respectivas células e também salvas em **PNG e PDF** em `resultados/modelagem_espectral_transferencia/figuras_relatorio/`. Os dados são conferidos antes da geração das Figuras 5 e 6; qualquer divergência relevante interrompe a execução.

In [ ]:
"""Figuras 1--6 do estudo dirigido OFC/EDFA.

Dados reais somente: as figuras 4, 5 e 6 lêem históricos, rótulos
TestGround e previsões auditadas; nenhuma série é sintetizada.
Figuras 1 a 3 sao esquemas conceituais. Gera PNG/PDF em arquivo temporario local
e depois copia ao Volume, evitando escrita aleatoria no Unity Catalog.

Execução em Databricks:
  python scripts/gerar_figuras.py --root /Volumes/workspace/default/ofc2026
Ou execute o conteúdo da cópia notebook/celula_26_figuras.py em uma célula.
"""
from __future__ import annotations

import argparse
from pathlib import Path
import tempfile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

N_CANAIS = 95
COLUNAS_LABEL = [f'calculated_gain_spectra_{i:02d}' for i in range(N_CANAIS)]
COLUNAS_MASK = [f'DUT_WSS_activated_channel_index_{i:02d}' for i in range(N_CANAIS)]


def salvar(fig, destino, nome):
    destino.mkdir(parents=True, exist_ok=True)
    # Gera em disco local: alguns sistemas de arquivo (Unity Catalog
    # Volumes) não suportam todas as operações de escrita do backend PDF.
    with tempfile.TemporaryDirectory(prefix='edfa_fig_') as temp:
        for ext in ('png', 'pdf'):
            origem = Path(temp) / f'{nome}.{ext}'
            fig.savefig(origem, dpi=240, bbox_inches='tight')
            shutil.copyfile(origem, destino / origem.name)
    plt.close(fig)
    print(f'Figura salva: {destino / (nome + ".png")}')


def bloco(ax, x, y, w, h, titulo, tamanho=11):
    patch = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.025',
                           linewidth=1.3, facecolor='none', edgecolor='0.30')
    ax.add_patch(patch)
    ax.text(x+w/2, y+h/2, titulo, ha='center', va='center', fontsize=tamanho)


def seta(ax, x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle='-|>',
                                 mutation_scale=14, linewidth=1.25))


def figura1(destino):
    """Esquema conceitual, não um modelo físico detalhado."""
    fig, ax = plt.subplots(figsize=(11,3.2))
    ax.set_xlim(0, 10); ax.set_ylim(0, 4); ax.axis('off')
    bloco(ax, .3, 1.6, 2, .9, 'Sinais WDM\n(entrada)')
    bloco(ax, 3.25, 1.6, 3.1, .9, 'Fibra dopada com érbio\n(EDFA)')
    bloco(ax, 7.2, 1.6, 2.4, .9, 'Sinais amplificados\n(saída)')
    bloco(ax, 3.75, 3.05, 2.1, .65, 'Bombeamento\nóptico', 10)
    seta(ax, 2.35, 2.05, 3.2, 2.05)
    seta(ax, 6.4, 2.05, 7.15, 2.05)
    seta(ax, 4.8, 3.02, 4.8, 2.55)
    ax.set_title('Figura 1 — Esquema conceitual de um EDFA', pad=12)
    salvar(fig, destino, 'figura_01_esquema_edfa')


def figura2(destino):
    fig, ax = plt.subplots(figsize=(11,4.8))
    ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')
    bloco(ax, .2, 3.75, 2.55, 1.1, 'COSMOS\n138.624 observações')
    bloco(ax, 3.7, 3.75, 2.4, 1.1, 'Pré-treinamento\nMLP (48 épocas)')
    bloco(ax, 7.0, 3.75, 2.7, 1.1, 'Checkpoint inicial\npré-treinado')
    bloco(ax, .2, 1.1, 2.55, 1.1, 'Challenge Train\n490 observações')
    bloco(ax, 3.7, 1.1, 2.4, 1.1, 'Ajuste fino\n(40 épocas)')
    bloco(ax, 7.0, 1.1, 2.7, 1.1, 'MLP final\n204 → 95')
    seta(ax, 2.8, 4.3, 3.65, 4.3)
    seta(ax, 6.15, 4.3, 6.95, 4.3)
    seta(ax, 8.35, 3.7, 8.35, 2.25)
    seta(ax, 2.8, 1.65, 3.65, 1.65)
    seta(ax, 6.15, 1.65, 6.95, 1.65)
    ax.text(5.0, .22, 'TestGround online: avaliação retrospectiva; não usado no ajuste de pesos',
            ha='center', fontsize=10)
    ax.set_title('Figura 2 — Fluxo experimental', pad=8)
    salvar(fig, destino, 'figura_02_pipeline')


def figura3(destino):
    fig, ax = plt.subplots(figsize=(12,3.7))
    ax.set_xlim(0, 13); ax.set_ylim(0, 4); ax.axis('off')
    blocos = [(0.15, 'Entradas\n204'),(2.2,'Linear + GELU\n256'),
              (4.45,'Linear + GELU\n256'),(6.7,'Linear + GELU\n128'),
              (8.95,'Resíduos\n95'),(11.1,'Ganho previsto\n95')]
    for x, s in blocos:
        bloco(ax, x, 1.5, 1.72, .9, s, 9.5)
    for (xa,_),(xb,_) in zip(blocos,blocos[1:]):
        seta(ax, xa+1.76, 1.95, xb-.06, 1.95)
    ax.text(10.05, .65, 'Somar referência linear de ganho e tilt',
            ha='center', fontsize=9)
    ax.text(3.3, 1.12, 'Dropout 0,10 nas duas primeiras camadas ocultas',
            ha='center', fontsize=9)
    ax.set_title('Figura 3 — Arquitetura MLP residual', pad=8)
    salvar(fig, destino, 'figura_03_arquitetura_mlp')


def figura4(destino, resultados):
    arquivos = [
        (resultados/'historico_mlp_v2.csv', 'mae_cosmos_val'),
        (resultados/'historico_pretrain_extended.csv', 'mae_val'),
        (resultados/'historico_pretrain_48ep.csv', 'mae_cosmos_val')
    ]
    partes = []
    for caminho, coluna in arquivos:
        if not caminho.exists():
            raise FileNotFoundError(f'Histórico necessário ausente: {caminho}')
        df = pd.read_csv(caminho)
        if not {'epoca', coluna}.issubset(df.columns):
            raise ValueError(f'Colunas inesperadas em {caminho}: {list(df.columns)}')
        partes.append(df[['epoca', coluna]].rename(columns={coluna: 'mae'}))
    hist = pd.concat(partes, ignore_index=True).sort_values('epoca')
    hist = hist.drop_duplicates('epoca', keep='last')
    if not set(range(1,49)).issubset(set(hist['epoca'].astype(int))):
        raise ValueError('Histórico incompleto: conferir épocas 1 a 48')
    fig, ax = plt.subplots(figsize=(9,4.2))
    ax.plot(hist['epoca'], hist['mae'], marker='.', linewidth=1.7)
    ax.axvline(8, linestyle='--', linewidth=1, alpha=.65)
    ax.axvline(24, linestyle='--', linewidth=1, alpha=.65)
    ax.set_xlabel('Época de pré-treinamento COSMOS')
    ax.set_ylabel('MAE de validação COSMOS (dB)')
    ax.set_title('Figura 4 — Evolução do MAE durante o pré-treinamento')
    ax.grid(alpha=.2)
    salvar(fig, destino, 'figura_04_pretrain_cosmos')


def carregar_validar(root, resultados):
    X = pd.read_csv(root/'Features/TestGround/test_features.csv')
    Y = pd.read_csv(root/'Features/TestGround/test_labels.csv')[COLUNAS_LABEL].to_numpy(float)
    P = pd.read_csv(resultados/'predicoes_mlp_v2_48ep_testground.csv')[COLUNAS_LABEL].to_numpy(float)
    M = X[COLUNAS_MASK].to_numpy(bool)
    if Y.shape != P.shape or P.shape != M.shape or P.shape != (21056,95):
        raise ValueError(f'Dimensões incompatíveis: Y={Y.shape}, P={P.shape}, M={M.shape}')
    if not (M.sum(axis=1)>0).all():
        raise ValueError('Observações sem canais ativos')
    if not (np.isfinite(Y[M]).all() and np.isfinite(P[M]).all()):
        raise ValueError('Valores não finitos em canais ativos')
    erro = (np.where(M, np.abs(Y-P), 0).sum(axis=1) / M.sum(axis=1))
    print(f'MAE local reproduzido: {erro.mean():.9f} dB')
    if not np.isclose(erro.mean(), 0.124232492, rtol=0, atol=1e-5):
        raise ValueError('O MAE não coincide com o resultado auditado; conferir arquivos!')
    cat = X['Category'].str.lower()
    for nome,n,mae_esperado in [('aging',4400,0.087264),
                                 ('shb',864,0.105130),
                                 ('unseen',15792,0.135578)]:
        sub = erro[cat.eq(nome).to_numpy()]
        if len(sub)!=n or not np.isclose(sub.mean(),mae_esperado,rtol=0,atol=1e-5):
            raise ValueError(f'Cenário {nome} inesperado: n={len(sub)}, MAE={sub.mean()}')
    usage = X['Usage'].astype(str)
    for nome,n,mae_esperado in [('Public',10528,0.127922),
                                 ('Private',10528,0.120543)]:
        sub = erro[usage.eq(nome).to_numpy()]
        if len(sub)!=n or not np.isclose(sub.mean(),mae_esperado,rtol=0,atol=1e-5):
            raise ValueError(f'Partição {nome} inesperada: n={len(sub)}, MAE={sub.mean()}')
    mae_canal_ativo = float(np.mean(np.abs(Y[M]-P[M])))
    if not np.isclose(mae_canal_ativo,0.106953,rtol=0,atol=1e-5):
        raise ValueError(f'MAE por canal ativo inesperado: {mae_canal_ativo}')
    print(f'MAE global por canal ativo: {mae_canal_ativo:.6f} dB')
    print('Conferência por cenário e Public/Private: aprovada')
    return X,Y,P,M,erro


def figura5(destino, X, Y, P, M, erro):
    """Escolhe observação com MAE próximo à mediana por cenário.

    Todos os pontos apresentados correspondem a canais ativos;
    não conecta pontos distantes, evitando interpolação inexistente.
    """
    categorias = X['Category'].str.lower().to_numpy()
    fig, eixos = plt.subplots(1,3,figsize=(14,4.3),sharey=False)
    escolhidos = []
    for ax,cat in zip(eixos, ('aging','shb','unseen')):
        idx_cat = np.flatnonzero(categorias == cat)
        if not len(idx_cat): raise ValueError(f'Categoria ausente: {cat}')
        # Se possível, evita uma única amostra ativa; caso não, mantém o registro.
        possiveis = idx_cat[M[idx_cat].sum(axis=1)>=2]
        if len(possiveis)==0: possiveis=idx_cat
        alvo = np.median(erro[possiveis])
        idx = int(possiveis[np.argmin(np.abs(erro[possiveis]-alvo))])
        canais = np.flatnonzero(M[idx])
        ax.scatter(canais,Y[idx,canais],s=40,marker='o',label='Observado')
        ax.scatter(canais,P[idx,canais],s=40,marker='x',label='Previsto')
        ax.set_xlim(-1,95)
        ax.set_xlabel('Índice do canal óptico')
        ax.set_title(f'{cat.upper()}\nMAE = {erro[idx]:.4f} dB',fontsize=10)
        ax.grid(alpha=.2)
        ax.legend(fontsize=8)
        escolhidos.append({'cenario':cat,'indice_linha_csv':idx,
                           'id':str(X.iloc[idx]['ID']) if 'ID' in X else '',
                           'ativos':len(canais),'mae_observacao':erro[idx]})
    eixos[0].set_ylabel('Ganho espectral (dB)')
    fig.suptitle('Figura 5 — Ganho observado e previsto apenas nos canais WSS ativos',fontsize=12)
    fig.tight_layout()
    salvar(fig, destino, 'figura_05_exemplos_canais_ativos')
    pd.DataFrame(escolhidos).to_csv(destino/'figura_05_indices_selecionados.csv',index=False)
    print('Observações representadas na Figura 5:')
    print(pd.DataFrame(escolhidos).to_string(index=False))


def figura6(destino, X, erro):
    """Boxplot da distribuicao do MAE por registro, como no relatorio.

    Hastes P5--P95, todos os pontos fora da faixa mostrados; valores
    nulos ou abaixo do limite positivo do eixo log nao sao desenhados.
    """
    categorias = X['Category'].str.lower().to_numpy()
    grupos = [erro[categorias==cat] for cat in ('aging','shb','unseen')]
    fig, ax = plt.subplots(figsize=(9,4.8))
    ax.boxplot(grupos, tick_labels=['Aging','SHB','Unseen'],
               whis=(5,95), showfliers=True,
               flierprops={'marker':'.','markersize':2,'alpha':.3})
    ax.set_yscale('log')
    ax.set_ylim(bottom=1e-5)
    ax.set_ylabel('MAE por observação (dB) — escala logarítmica')
    ax.set_title('Figura 6 — Distribuição do MAE por cenário')
    ax.grid(axis='y',alpha=.25,which='both')
    fig.tight_layout()
    salvar(fig,destino,'figura_06_distribuicao_mae')
    resumo = []
    for cat,v in zip(('aging','shb','unseen'),grupos):
        resumo.append({'cenario':cat,'n':len(v),'mae_medio':np.mean(v),
                       'mediana':np.median(v),'p95':np.quantile(v,.95),
                       'p99':np.quantile(v,.99),'maximo':np.max(v),
                       'erros_iguais_zero':int((v==0).sum())})
    tab = pd.DataFrame(resumo)
    tab.to_csv(destino/'figura_06_resumo_mae.csv',index=False)
    print('Resumo por cenário:')
    print(tab.to_string(index=False))



from IPython.display import display, Image

FIG_ROOT = Path('/Volumes/workspace/default/ofc2026')
FIG_RESULTS = FIG_ROOT / 'resultados/modelagem_espectral_transferencia'
FIG_OUT = FIG_RESULTS / 'figuras_relatorio'
FIG_OUT.mkdir(parents=True, exist_ok=True)

def mostrar_figura(nome):
    display(Image(filename=str(FIG_OUT / f'{nome}.png')))

print('Arquivos PNG/PDF serão criados em:',FIG_OUT)


### Figura 1 — Esquema simplificado do EDFA

In [ ]:
figura1(FIG_OUT)
mostrar_figura("figura_01_esquema_edfa")

### Figura 2 — Fluxo metodológico COSMOS → Challenge → TestGround

In [ ]:
figura2(FIG_OUT)
mostrar_figura("figura_02_pipeline")

### Figura 3 — Arquitetura MLP residual 204 → 256 → 256 → 128 → 95

In [ ]:
figura3(FIG_OUT)
mostrar_figura("figura_03_arquitetura_mlp")

### Figura 4 — Histórico de validação COSMOS nas 48 épocas

In [ ]:
figura4(FIG_OUT, FIG_RESULTS)
mostrar_figura("figura_04_pretrain_cosmos")

### Figura 5 — Ganhos observados e previstos nos três cenários

In [ ]:
FIG_DADOS = carregar_validar(FIG_ROOT, FIG_RESULTS)
figura5(FIG_OUT, *FIG_DADOS)
mostrar_figura("figura_05_exemplos_canais_ativos")

### Figura 6 — Distribuição do MAE por cenário

In [ ]:
if "FIG_DADOS" not in globals():
    FIG_DADOS = carregar_validar(FIG_ROOT, FIG_RESULTS)
figura6(FIG_OUT, FIG_DADOS[0], FIG_DADOS[4])
mostrar_figura("figura_06_distribuicao_mae")

## 11. Validação independente dos seis gráficos e resultados

Esta célula pode ser executada **depois da geração das figuras, mesmo com o estado da sessão reiniciado**. Ela lê os CSVs diretamente do Volume, sem depender da variável `FIG_DADOS`. Confere os 12 arquivos PNG/PDF, as dimensões e os MAEs global, por canal ativo, por cenário e por partição Public/Private. Não treina modelos. A inspeção visual das figuras continua necessária.


In [ ]:
# 11 - VALIDACAO INDEPENDENTE DAS FIGURAS E DOS RESULTADOS
# Pode ser executada em notebook Python novo, sem FIG_DADOS ou outras variáveis prévias.
from pathlib import Path
import numpy as np
import pandas as pd

RAIZ = Path('/Volumes/workspace/default/ofc2026')
RESULTADOS = RAIZ / 'resultados/modelagem_espectral_transferencia'
FIGURAS = RESULTADOS / 'figuras_relatorio'

nomes_figuras = [
    'figura_01_esquema_edfa',
    'figura_02_pipeline',
    'figura_03_arquitetura_mlp',
    'figura_04_pretrain_cosmos',
    'figura_05_exemplos_canais_ativos',
    'figura_06_distribuicao_mae',
]

for nome in nomes_figuras:
    for extensao in ('png', 'pdf'):
        arquivo = FIGURAS / f'{nome}.{extensao}'
        if not arquivo.is_file() or arquivo.stat().st_size == 0:
            raise FileNotFoundError(
                f'Figura ausente/vazia: {arquivo}. Execute a seção 10 das figuras.'
            )
print('1. Figuras: seis PNGs e seis PDFs presentes e não vazios. OK')

colunas_ganho = [f'calculated_gain_spectra_{i:02d}' for i in range(95)]
colunas_mascara = [f'DUT_WSS_activated_channel_index_{i:02d}' for i in range(95)]

entrada = RAIZ / 'Features/TestGround/test_features.csv'
rotulos = RAIZ / 'Features/TestGround/test_labels.csv'
previsoes = RESULTADOS / 'predicoes_mlp_v2_48ep_testground.csv'

for arquivo in (entrada, rotulos, previsoes):
    if not arquivo.is_file():
        raise FileNotFoundError(f'Dados necessários ausentes: {arquivo}')

X = pd.read_csv(entrada)
Y = pd.read_csv(rotulos, usecols=colunas_ganho)[colunas_ganho].to_numpy(float)
P = pd.read_csv(previsoes, usecols=colunas_ganho)[colunas_ganho].to_numpy(float)
M = X[colunas_mascara].to_numpy(bool)

if X.shape[0] != 21056 or Y.shape != (21056, 95) or P.shape != Y.shape or M.shape != Y.shape:
    raise AssertionError(f'Dimensões incorretas: X={X.shape}, Y={Y.shape}, P={P.shape}, M={M.shape}')

ativos_por_registro = M.sum(axis=1)
if np.any(ativos_por_registro == 0):
    raise AssertionError('Há registros sem canais ativos')
if int(M.sum()) != 262144:
    raise AssertionError(f'Total de canais ativos inesperado: {int(M.sum())}')
if not (np.isfinite(Y[M]).all() and np.isfinite(P[M]).all()):
    raise AssertionError('Ganho não finito em canal ativo')

mae_por_registro = np.where(M, np.abs(Y-P), 0.0).sum(axis=1) / ativos_por_registro

def conferir(rotulo, obtido, esperado, tol=1e-5):
    if not np.isclose(obtido, esperado, rtol=0, atol=tol):
        raise AssertionError(f'{rotulo}: {obtido:.9f} dB, esperado {esperado:.9f} dB')
    print(f'{rotulo}: {obtido:.6f} dB — OK')

conferir('2. MAE global por observação', float(mae_por_registro.mean()), 0.124232492)
conferir('3. MAE por canal ativo', float(np.abs(Y[M]-P[M]).mean()), 0.106953)

for categoria, n, esperado in [
    ('aging', 4400, 0.087264),
    ('shb', 864, 0.105130),
    ('unseen', 15792, 0.135578),
]:
    selecionadas = X['Category'].astype(str).str.lower().eq(categoria).to_numpy()
    if int(selecionadas.sum()) != n:
        raise AssertionError(f'Cenário {categoria}: número inesperado de observações')
    conferir(f'4. {categoria.upper()}', float(mae_por_registro[selecionadas].mean()), esperado)

for uso, esperado in [('public', 0.127922), ('private', 0.120543)]:
    selecionadas = X['Usage'].astype(str).str.lower().eq(uso).to_numpy()
    if int(selecionadas.sum()) != 10528:
        raise AssertionError(f'Partição {uso}: número inesperado de observações')
    conferir(f'5. {uso.title()}', float(mae_por_registro[selecionadas].mean()), esperado)

print('VALIDAÇÃO INDEPENDENTE CONCLUÍDA: resultados numéricos e arquivos gráficos conferidos.')
print('Observação: é necessário verificar visualmente a legibilidade das imagens.')
